# nuScenes pretrained inference (LiDAR + camera) for radar late fusion

Runs public MMDetection3D checkpoints on the nuScenes samples you have, and writes
nuScenes-format detection JSONs that `scripts/fuse.py` consumes on your PC.

**Before running:** Runtime > Change runtime type > GPU (T4). In the settings cell, paste the download links for the metadata and one blob, obtained from nuscenes.org while logged in (the links are tied to your account and must not be shared), or point the fields at copies on Google Drive.

Runs in order: install (10 to 15 min), data (20 to 40 min), infos (10 min),
LiDAR inference (10 min), camera inference (30 to 60 min).

In [ ]:
#@title Settings
META_TGZ = "PASTE_YOUR_NUSCENES_LINK_FOR_v1.0-trainval_meta.tgz"   #@param {type:"string"}   URL or Drive path of v1.0-trainval_meta.tgz
BLOB_TGZ = "PASTE_YOUR_NUSCENES_LINK_FOR_v1.0-trainvalNN_blobs.tgz"   #@param {type:"string"}   URL or Drive path of one trainval blob (blob 01 = daytime scenes, blob 10 = night and rain validation scenes)
USE_MINI = False  #@param {type:"boolean"}  set True and put v1.0-mini.tgz in BLOB_TGZ for a quick test
LIDAR_KEEP_RATIO = 1.0  #@param {type:"number"}  1.0 = clean; 0.25 = keep 25% of LiDAR points (uniform degradation run)
LIDAR_RUNS = '[{"tag": ""}]'  #@param {type:"string"}  JSON list of LiDAR runs for this session: {"tag","sweeps","keep_ratio","range_keep"}; sweeps = past sweeps added to the keyframe (checkpoint: 9); range_keep = keep probability within 20 m, 20-30, 30-40, beyond 40 m, e.g. "1.0,0.5,0.2,0.1"
SKIP_CAMERA = False  #@param {type:"boolean"}  True = LiDAR only (use for the degradation run, the camera detections already exist)
OUT_DIR = "/content/out"
DATA_ROOT = "/content/data/nuscenes"
VERSION = "v1.0-mini" if USE_MINI else "v1.0-trainval"
import os; os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(DATA_ROOT, exist_ok=True)
print(VERSION, "keep ratio", LIDAR_KEEP_RATIO)

In [ ]:
#@title Install MMDetection3D in a Python 3.11 environment (Colab's default Python is too new for these libraries)
%cd /content
!pip install -q uv
!uv venv --python 3.11 /content/venv -q
PY = "/content/venv/bin/python"
!uv pip install --python $PY -q "setuptools<70" wheel "numpy<2" torch==2.1.2 torchvision==0.16.2
!uv pip install --python $PY -q mmengine==0.10.4 mmdet==3.2.0 nuscenes-devkit openmim
!uv pip install --python $PY -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
!git clone -q -b v1.4.0 https://github.com/open-mmlab/mmdetection3d.git
%cd /content/mmdetection3d
!uv pip install --python $PY -q -e . --no-build-isolation
!$PY -c "import numba, mmdet3d.datasets; print('mmdet3d deps ok')"
!$PY -c "import torch, mmcv, mmdet, mmdet3d; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mmcv', mmcv.__version__, 'mmdet3d', mmdet3d.__version__)"

In [ ]:
#@title Get the data (streaming extract, no temporary archive on disk)
import os
os.chdir(DATA_ROOT)
# Camera sweep images (non-keyframe frames) are not used by CenterPoint or FCOS3D; skipping them
# saves roughly a third of the disk, which blob 10 (41.7 GB) needs to fit in Colab's 112 GB.
EXCLUDE = "--exclude='sweeps/CAM_*'"   # quoted so the shell does not expand the pattern
def stream_extract(src, exclude=""):
    if src.startswith("http"):
        os.system(f'wget -qO- "{src}" | tar -xzf - {exclude}')
    else:
        os.system(f'tar -xzf "{src}" {exclude}')
if not USE_MINI:
    stream_extract(META_TGZ)
stream_extract(BLOB_TGZ, EXCLUDE)
print(os.listdir(DATA_ROOT))
!df -h /content | tail -1

In [ ]:
#@title Build info files (only scenes whose files are present)
import os
os.chdir("/content/mmdetection3d")
os.environ["PYTHONPATH"] = "/content/mmdetection3d"
os.makedirs("/content/mmdetection3d/data", exist_ok=True)
if not os.path.exists("/content/mmdetection3d/data/nuscenes"):
    os.symlink(DATA_ROOT, "/content/mmdetection3d/data/nuscenes")  # update_infos_to_v2 hardcodes ./data/nuscenes
script = r'''
import os, sys
sys.path.insert(0, "/content/mmdetection3d")
import mmengine
from tools.dataset_converters import nuscenes_converter as nc
from tools.dataset_converters.update_infos_to_v2 import update_pkl_infos
root, version, prefix = sys.argv[1], sys.argv[2], "nuscenes"
AVAIL = set()
def get_available_scenes(nusc):
    avail = []
    for scene in nusc.scene:
        sample = nusc.get("sample", scene["first_sample_token"])
        sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
        if os.path.exists(os.path.join(nusc.dataroot, sd["filename"])):
            avail.append(scene); AVAIL.add(scene["token"])
    print("scenes with files on disk:", len(avail))
    return avail
nc.get_available_scenes = get_available_scenes
_orig_track = mmengine.track_iter_progress
def track_filtered(items, *a, **k):
    items = [s for s in items if not (isinstance(s, dict) and "scene_token" in s) or s["scene_token"] in AVAIL]
    return _orig_track(items, *a, **k)
mmengine.track_iter_progress = track_filtered
nc.create_nuscenes_infos(root, prefix, version=version, max_sweeps=10)
for split in ("train", "val"):
    update_pkl_infos("nuscenes", out_dir=root, pkl_path=os.path.join(root, f"{prefix}_infos_{split}.pkl"))
print("done")
'''
open("/content/make_infos.py", "w").write(script)
!$PY /content/make_infos.py $DATA_ROOT $VERSION
!ls -la $DATA_ROOT/*.pkl

In [ ]:
#@title Download checkpoints (URLs read from the repo metafiles)
import os, glob, subprocess, yaml
os.chdir("/content/mmdetection3d")
LIDAR_CFG = "centerpoint_pillar02_second_secfpn_head-circlenms_8xb4-cyclic-20e_nus-3d"
CAM_CFG = "fcos3d_r101-caffe-dcn_fpn_head-gn_8xb2-1x_nus-mono3d_finetune"
def ckpt_url(cfg_name):
    meta = yaml.safe_load(open(f"configs/{cfg_name.split('_')[0]}/metafile.yml"))
    for m in meta["Models"]:
        if m["Config"].endswith(cfg_name + ".py"):
            return m["Weights"]
    raise KeyError(cfg_name)
os.makedirs("checkpoints", exist_ok=True)
for cfg in (LIDAR_CFG, CAM_CFG):
    url = ckpt_url(cfg); print(url)
    subprocess.run(["wget", "-q", "-nc", "-P", "checkpoints", url], check=True)
LIDAR_CKPT = glob.glob("checkpoints/centerpoint*.pth")[0]
CAM_CKPT = glob.glob("checkpoints/fcos3d*.pth")[0]
print(LIDAR_CKPT); print(CAM_CKPT)

In [ ]:
#@title Run inference on val and train samples, write nuScenes-format JSON
%cd /content/mmdetection3d
import glob, json, os, shutil, subprocess
# range-dependent dropout transform, registered through custom_imports so tools/test.py can build it
rd = r"""
import numpy as np
from mmcv.transforms import BaseTransform
from mmdet3d.registry import TRANSFORMS


@TRANSFORMS.register_module()
class RangeDropout(BaseTransform):
    # Keeps each point with a probability that depends on its horizontal range from the sensor:
    # keep[0] within edges[0] metres, keep[1] up to edges[1], ... keep[-1] beyond edges[-1].
    # Models a LiDAR whose returns fade with distance (weather, low reflectivity), unlike a
    # uniform subsample. One random stream per run (seeded), applied after sweep aggregation.
    def __init__(self, edges=(20.0, 30.0, 40.0), keep=(1.0, 0.5, 0.2, 0.1), seed=0):
        self.edges = np.asarray(edges, dtype=np.float32)
        self.keep = np.asarray(keep, dtype=np.float32)
        assert len(self.keep) == len(self.edges) + 1
        self.rng = np.random.default_rng(seed)

    def transform(self, results):
        pts = results["points"]
        xyz = pts.coord.numpy()
        r = np.hypot(xyz[:, 0], xyz[:, 1])
        p = self.keep[np.digitize(r, self.edges)]
        idx = np.flatnonzero(self.rng.random(len(p)) < p)
        results["points"] = pts[idx]
        return results
"""
open("/content/range_dropout.py", "w").write(rd)
prep = r"""
import json, os, sys
from mmengine.config import Config
cfg_name, split, run_json, data_root, out_dir = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5]
run = json.loads(run_json)
tag_suffix = ("_" + run["tag"]) if run.get("tag") else ""
sweeps = run.get("sweeps")
keep_ratio = float(run.get("keep_ratio", 1.0))
range_keep = run.get("range_keep", "")
cfg = Config.fromfile(f"configs/{cfg_name.split('_')[0]}/{cfg_name}.py")
ann = f"nuscenes_infos_{split}.pkl"
cfg.test_dataloader.dataset.ann_file = ann
cfg.test_dataloader.dataset.data_root = data_root
cfg.test_evaluator.ann_file = os.path.join(data_root, ann)
cfg.test_evaluator.data_root = data_root
cfg.test_evaluator.format_only = True
tag = f"{cfg_name.split('_')[0]}_{split}{tag_suffix}"
cfg.test_evaluator.jsonfile_prefix = os.path.join(out_dir, tag)
pipe = cfg.test_dataloader.dataset.pipeline
if sweeps is not None:
    for t in pipe:
        if t["type"] == "LoadPointsFromMultiSweeps":
            t["sweeps_num"] = int(sweeps)
idx = max(i for i, t in enumerate(pipe) if t["type"] in ("LoadPointsFromFile", "LoadPointsFromMultiSweeps"))
if range_keep:
    keep = [float(x) for x in range_keep.split(",")]
    pipe.insert(idx + 1, dict(type="RangeDropout", edges=(20.0, 30.0, 40.0), keep=keep, seed=0))
    idx += 1
    cfg.custom_imports = dict(imports=["range_dropout"], allow_failed_imports=False)
if keep_ratio < 1.0:
    pipe.insert(idx + 1, dict(type="PointSample", num_points=keep_ratio))
path = f"/content/{tag}.py"; cfg.dump(path)
print(path); print(tag)
"""
open("/content/prepare_cfg.py", "w").write(prep)
os.environ["PYTHONPATH"] = "/content/mmdetection3d:/content"

runs = json.loads(LIDAR_RUNS)
if LIDAR_KEEP_RATIO < 1.0 and runs == [{"tag": ""}]:
    runs = [{"tag": f"keep{int(LIDAR_KEEP_RATIO*100)}", "keep_ratio": LIDAR_KEEP_RATIO}]
RUN_TAGS = [r.get("tag", "") for r in runs]
jobs = [(LIDAR_CFG, LIDAR_CKPT, r) for r in runs] + ([] if SKIP_CAMERA else [(CAM_CFG, CAM_CKPT, {"tag": ""})])
for cfg_name, ckpt, run in jobs:
    for split in ("val", "train"):
        out = subprocess.run([PY, "/content/prepare_cfg.py", cfg_name, split, json.dumps(run), DATA_ROOT, OUT_DIR],
                             capture_output=True, text=True)
        if out.returncode != 0:
            print(out.stderr); raise SystemExit("config preparation failed")
        cfg_path, tag = out.stdout.strip().splitlines()[-2:]
        print("=== running", tag, run)
        !$PY tools/test.py $cfg_path $ckpt --work-dir /content/work/$tag
        found = glob.glob(os.path.join(OUT_DIR, tag, "**", "results_nusc.json"), recursive=True)
        print("results:", found)
        if found:
            shutil.copy(found[0], os.path.join(OUT_DIR, f"{tag}.json"))
print(os.listdir(OUT_DIR))

In [ ]:
#@title Merge train+val per detector and package for download
import json, os
def merge(prefix, suffix=""):
    out = {"meta": None, "results": {}}
    for split in ("val", "train"):
        p = os.path.join(OUT_DIR, f"{prefix}_{split}{suffix}.json")
        if not os.path.exists(p):
            print("missing", p); continue
        d = json.load(open(p)); out["meta"] = d["meta"]; out["results"].update(d["results"])
    dst = os.path.join(OUT_DIR, f"{prefix}{suffix}.json"); json.dump(out, open(dst, "w"))
    print(dst, len(out["results"]), "samples")
for tag in RUN_TAGS:
    merge("centerpoint", ("_" + tag) if tag else "")
if not SKIP_CAMERA:
    merge("fcos3d")
!cd $OUT_DIR && zip -q -j detections.zip centerpoint*.json fcos3d*.json && ls -la detections.zip
from google.colab import files
files.download(os.path.join(OUT_DIR, "detections.zip"))

On your PC, unzip `detections.zip` into `D:\moath\radar-fusion\detections\` and run:

```
.venv\Scripts\python scripts\fuse.py --dataroot data\nuscenes --lidar detections\centerpoint.json --camera detections\fcos3d.json --out results\fused
.venv\Scripts\python scripts\evaluate.py --dataroot data\nuscenes --results results\fused
```

For the degradation rows, rerun this notebook with `SKIP_CAMERA = True` and either `LIDAR_KEEP_RATIO = 0.25` (uniform thinning, output `centerpoint_keep25.json`) or a `LIDAR_RUNS` list such as
`[{"tag": "sw1", "sweeps": 1}, {"tag": "sw1_far", "sweeps": 1, "range_keep": "1.0,0.5,0.2,0.1"}]`
(fewer sweeps, so the detector's own velocity estimate degrades, and a range-dependent dropout on top; outputs `centerpoint_sw1.json` and `centerpoint_sw1_far.json`). Fuse each with `--lidar detections\centerpoint_<tag>.json --out results\fused_<tag>`.